In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_groq import ChatGroq
groq_api_key=os.getenv("GROQ_API_KEY")
llm=ChatGroq(model="llama-3.1-8b-instant",groq_api_key=groq_api_key)


In [127]:
import bs4
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://www.coursera.org/articles/what-is-artificial-intelligence?utm_medium=sem&utm_source=gg&utm_campaign=b2c_india_x_multi_ftcof_career-academy_cx_dr_bau_gg_pmax_gc_in_all_m_hyb_24-03_desktop&campaignid=21104989118&adgroupid=&device=c&keyword=&matchtype=&network=x&devicemodel=&creativeid=&assetgroupid=6544944776&targetid=&extensionid=&placement=&gad_source=1&gad_campaignid=21104990591&gbraid=0AAAAADdKX6Z5WlJMVIEBGzwdyuJFe11rK&gclid=CjwKCAiAybfLBhAjEiwAI0mBBh6JTaUZS9PzwS_I9RZTfFZ75rEWIojg2MMzUk4znOli0cJrlpebkhoCEAYQAvD_BwE"
)


docs = loader.load()


In [114]:
print("DOCS COUNT:", len(docs))
print(docs[:1])

DOCS COUNT: 1
[Document(metadata={'source': 'https://www.digitalocean.com/community/tutorials/model-context-protocol', 'title': 'MCP 101: An Introduction to Model Context Protocol | DigitalOcean', 'description': 'The goal of this article is to give readers an introduction to Model Context Protocol (MCP). ', 'language': 'en'}, page_content="MCP 101: An Introduction to Model Context Protocol | DigitalOceanBlogDocsGet SupportContact SalesDigitalOceanProductsFeatured ProductsDropletsScalable virtual machinesKubernetesScale more effectivelyGradient™ AI Inference CloudBuild and scale with AICloudwaysManaged cloud hostingApp PlatformGet apps to market fasterManaged DatabasesFully-managed database hostingComputeDropletsKubernetesCPU-Optimized DropletsFunctionsApp PlatformGradient™ AI Inference CloudGPU Droplets1-Click ModelsPlatformBare Metal GPUsBackups & SnapshotsBackupsSnapshotsSnapShooterNetworkingVirtual Private Cloud (VPC)Partner Network ConnectCloud FirewallsLoad BalancersDNSDDoS Protec

In [130]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


In [126]:
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [131]:
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
splits=text_splitter.split_documents(docs)
vectorstore=Chroma.from_documents(documents=splits,embedding=embeddings)
retriever=vectorstore.as_retriever()



In [132]:
system_prompt="you are an helping assisstant so give answers from the given context ."
prompt=ChatPromptTemplate.from_messages(
                                    [
                                        ("system",system_prompt,),
                                        ("human","{input}")
                                    ]
)

In [133]:
from operator import itemgetter
rag_chain=(
    {
       "context":itemgetter("input")| retriever,
       "input":itemgetter("input")
    }
|prompt
|llm
)

In [135]:
response=rag_chain.invoke({"input":"What is AI?"})
response.content

'AI, or Artificial Intelligence, refers to the simulation of human intelligence in machines that are programmed to think and learn like humans. The term can also be applied to any machine that exhibits traits associated with a human mind such as learning and problem-solving.\n\nAI technology is divided into two types: \n\n1. **Narrow or Weak AI**: This type of AI is designed and trained for a particular task. It can perform a specific set of functions or tasks, but it is not capable of general reasoning or independent thinking.\n\n2. **General or Strong AI**: This type of AI is a hypothetical AI system that would possess the ability to understand, learn, and apply knowledge in a wide range of tasks, similar to human intelligence.\n\nAI has many applications, including:\n\n- Virtual assistants\n- Image and speech recognition\n- Natural Language Processing (NLP)\n- Predictive analytics\n- Robotics\n- Autonomous vehicles\n- Healthcare and medical diagnosis\n\nThe development of AI has bee